# Phase 2E — Versioned report corrections and provenance review

Reuse Phase 2D verified statistics. No dataset loading, training, inference, tuning or regenerated data. Original reports and numerical artifacts stay unchanged. Checkpoint inspection uses fake tensors to read metadata without constructing models or loading tensor storage. This notebook records Phase 2E work only; it is not a recovered Phase 2C notebook.


In [1]:
from pathlib import Path
import json, hashlib, csv, subprocess, re
import torch
from torch._subclasses.fake_tensor import FakeTensorMode
ROOT=Path.cwd()
if ROOT.name!="bilstm new updated":
    ROOT=ROOT/"bilstm new updated"
ROOT=ROOT.resolve()
RES=ROOT/"phase2_bilstm_results"
OUT=RES/"gemini_verification"
HUB=RES/"huber_experiment"
DIAG=RES/"error_diagnosis"
def readj(p):
    return json.loads(p.read_text(encoding="utf-8"))
def sha(p):
    h=hashlib.sha256()
    with p.open("rb") as f:
        for block in iter(lambda:f.read(1024*1024),b""):
            h.update(block)
    return h.hexdigest()
verified=readj(OUT/"verification_metrics.json")
with (OUT/"discrepancy_table.csv").open(encoding="utf-8",newline="") as f:
    discrepancies=list(csv.DictReader(f))
features=verified["phase2c_features"]["statistics"]
table_checks=verified["phase2c_temporal"]["feature_report_table_checks"]
corrections=[x for x in table_checks if not x["match"]]
assert len(corrections)==19 and len(features)==10
# All inspected sources protected, including data and checkpoints: hashes only, no dataset reads into arrays.
protected={rel:sha(ROOT/rel) for rel in verified["source_hashes_after"]}
assert all(protected[k]==v for k,v in verified["source_hashes_after"].items()),"Sources changed since independent verification"
protected.update({str((OUT/n).relative_to(ROOT)):sha(OUT/n) for n in
                  ["verification_report.md","verification_metrics.json","discrepancy_table.csv","feature_verification.csv"]})
queries=[
 ["rg","--files","--hidden",str(ROOT),"-g","*phase2c*","-g","*Phase2C*","-g","*feature*temporal*.ipynb","-g","!**/.training-runtime/**"],
 ["rg","-l","-i","--hidden","-g","*.ipynb","-g","!**/.training-runtime/**",
  "-g","!phase2d_gemini_verification.ipynb","-g","!phase2e_report_corrections.ipynb",
  "Phase 2C|Feature Informativeness|Feature.{0,30}Temporal.{0,20}Audit",str(ROOT)]
]
searches=[]
for args in queries:
    result=subprocess.run(args,capture_output=True,text=True,encoding="utf-8",errors="replace")
    assert result.returncode in [0,1],result.stderr
    searches.append(dict(command=args,exit_code=result.returncode,matches=result.stdout.splitlines()))
# Only immediate workspace and named checkpoint directories outside the project search.
extra=[]
for base in [ROOT.parent,ROOT.parent/".ipynb_checkpoints",ROOT/".ipynb_checkpoints"]:
    extra.append(dict(directory=str(base),exists=base.exists(),
                      matches=[str(p) for p in base.glob("*phase2c*")] if base.exists() else []))
nb=readj(ROOT/"phase2b_huber_loss_experiment.ipynb")
notebook_log=[]
for i,c in enumerate(nb["cells"]):
    if c["cell_type"]!="code":
        continue
    streams=[]
    for o in c.get("outputs",[]):
        if o.get("output_type")=="stream":
            t=o.get("text","")
            streams.append("".join(t) if isinstance(t,list) else t)
    text="\n".join(streams)
    notebook_log.append(dict(cell=i,execution_count=c.get("execution_count"),
        metadata=c.get("metadata",{}),errors=[o.get("ename") for o in c.get("outputs",[]) if o.get("output_type")=="error"],
        log_excerpt=text[:900] if i!=5 else "\n".join(text.splitlines()[-3:])))
meta={}
safe_keys=["hyperparameters","model_config","dataset_version","dataset_sha256","consumer_source_sha256",
           "augmentation_variant","target_definition","best_epoch","validation_mae","seed","parameter_count",
           "initialization_sha256","optimizer","loss","epoch","best_mae","stale","elapsed"]
for label,path in [("delta1_best",RES/"variants/C/best.pt"),("delta2_best",HUB/"best.pt"),("delta2_last",HUB/"last.pt")]:
    item={"path":str(path.relative_to(ROOT)),"sha256":sha(path)}
    try:
        with FakeTensorMode():
            ck=torch.load(path,map_location="cpu",weights_only=True,mmap=True)
        item["keys"]=sorted(ck.keys())
        item["saved_metadata"]={k:ck[k] for k in safe_keys if k in ck and isinstance(ck[k],(str,int,float,bool,list,dict,type(None)))}
        item["saved_metadata"].pop("optimizer",None) if isinstance(item["saved_metadata"].get("optimizer"),dict) else None
        opt=ck.get("optimizer_state",ck.get("optimizer"))
        if isinstance(opt,dict) and "param_groups" in opt:
            item["optimizer_param_groups"]=[{k:v for k,v in g.items() if k!="params"} for g in opt["param_groups"]]
        item["inspection"]="PASS: weights_only fake tensors; tensor storage not materialized; no model constructed"
        del ck
    except Exception as e:
        item["inspection"]="UNKNOWN"
        item["error"]=type(e).__name__+": "+str(e)
    meta[label]=item
# Inspect the original last checkpoint only for optimizer metadata comparison.
path=RES/"variants/C/last.pt"
protected[str(path.relative_to(ROOT))]=sha(path)
try:
    with FakeTensorMode():
        ck=torch.load(path,map_location="cpu",weights_only=True,mmap=True)
    opt=ck.get("optimizer_state",ck.get("optimizer"))
    meta["delta1_last"]={"path":str(path.relative_to(ROOT)),"sha256":sha(path),"keys":sorted(ck),
        "optimizer_param_groups":[{k:v for k,v in g.items() if k!="params"} for g in opt["param_groups"]] if isinstance(opt,dict) and "param_groups" in opt else {}}
    del ck
except Exception as e:
    meta["delta1_last"]={"inspection":"UNKNOWN","error":str(e)}
provenance={"date":"2026-11-10","stage":"Phase 2E, not recovered Phase 2C execution",
    "targeted_searches":searches,"additional_search_locations":extra,
    "phase2b_notebook_metadata":nb.get("metadata",{}),"saved_notebook_log_excerpts":notebook_log,
    "checkpoint_metadata":meta,"protected_hashes_before":protected}
print("Search matches:",[s["matches"] for s in searches])
for label,m in meta.items():
    print(label,json.dumps(m.get("saved_metadata",m.get("optimizer_param_groups",m)),ensure_ascii=True)[:2400])


Search matches: [['C:\\Users\\choud\\Desktop\\Minor project\\bilstm new updated\\phase2_bilstm_results\\error_diagnosis\\phase2c_feature_informativeness_report.md', 'C:\\Users\\choud\\Desktop\\Minor project\\bilstm new updated\\phase2_bilstm_results\\error_diagnosis\\phase2c_feature_audit.json'], ['C:\\Users\\choud\\Desktop\\Minor project\\bilstm new updated\\phase2a_bilstm_error_diagnosis.ipynb']]
delta1_best {"hyperparameters": {"hidden_size": 128, "layers": 2, "dropout": 0.3144753602375439, "learning_rate": 0.0004549047669323766, "weight_decay": 0.00012155233771053111, "batch_size": 128}, "model_config": {"hidden_size": 128, "layers": 2, "dropout": 0.3144753602375439}, "dataset_version": "v2_timestamp_fixed", "dataset_sha256": "97d357f71d6f83e3c7cbb5a163ca42b0ed40fbb2fc859c9e2076445c661c934f", "consumer_source_sha256": "5975497597e70a89b9f7009b861963cb23b4ecc7f6f2b99db8273920489b3058", "augmentation_variant": "C", "target_definition": {"run_id": "945fdfac7b47cfb0", "primary": "Next 

In [2]:
# Review the only alternate notebook hit; inspect saved metadata rather than rerunning code.
candidate_reviews=[]
for candidate in searches[1]["matches"]:
    p=Path(candidate); n=readj(p)
    title=next(("".join(c["source"])[:220] for c in n["cells"] if c["cell_type"]=="markdown"),"")
    matches=[line.strip()[:250] for c in n["cells"] for line in "".join(c["source"]).splitlines()
             if re.search(r"Phase 2C|Feature Informativeness|Feature.{0,30}Temporal.{0,20}Audit",line,re.I)]
    candidate_reviews.append(dict(path=str(p.relative_to(ROOT)),title=title,matching_lines=matches[:5],
                                  classification="Phase 2A artifact, not recovered Phase 2C execution"))
provenance["candidate_reviews"]=candidate_reviews
a=meta["delta1_best"]["saved_metadata"]; b=meta["delta2_best"]["saved_metadata"]
equal_fields=["hyperparameters","model_config","dataset_version","dataset_sha256","augmentation_variant",
              "target_definition","seed","parameter_count","initialization_sha256"]
saved_controls={k:a[k]==b[k] for k in equal_fields}
optimizer_equal=meta["delta1_last"].get("optimizer_param_groups")==meta["delta2_last"].get("optimizer_param_groups")
assert all(saved_controls.values()) and optimizer_equal and meta["delta2_last"].get("optimizer_param_groups")
for label,metadata in [("delta1",a),("delta2",b)]:
    row=verified["phase2b"]["best_history_rows"][label]
    assert metadata["best_epoch"]==row["epoch"] and abs(metadata["validation_mae"]-row["val_mae"])<1e-12
provenance["new_evidence"]={
    "matching_checkpoint_config_fields":saved_controls,"optimizer_groups_equal":optimizer_equal,
    "losses":[a["loss"],b["loss"]],"best_epochs":[a["best_epoch"],b["best_epoch"]],
    "best_epoch_val_mae_match_history":True,
    "delta2_last_epoch":meta["delta2_last"]["saved_metadata"]["epoch"],
    "delta2_stale_epochs":meta["delta2_last"]["saved_metadata"]["stale"],
    "interpretation":"Serialized controls and selection metadata support the comparison. Complete historical runtime equivalence and absolute test non-access remain UNKNOWN.",
    "missing":"Original Phase2C execution; full delta2 runtime/environment snapshot and independent access log; delta2 checkpoint consumer_source_sha256."}
assert not (ROOT/"phase2c_feature_temporal_audit.ipynb").exists()
def table(headers,rows):
    return "\n".join(["| "+" | ".join(headers)+" |","|"+"|".join(["---"]*len(headers))+"|"]+
                     ["| "+" | ".join(str(v) for v in row)+" |" for row in rows])
m=verified["phase2b"]["metrics"]; changes=verified["phase2b"]["changes"]
b_rows=[]
for display,field in [("Validation MAE","mae"),("Validation RMSE","rmse"),("Validation R2","r2"),
                     ("Validation Huber (own delta)","huber"),("Prediction mean","pred_mean"),
                     ("Prediction SD","pred_std"),("ML>=4.5 MAE (75 events)","high_mae"),
                     ("ML<3 MAE (195 events)","low_mae"),("Clean train MAE, logged","clean_train_mae_logged"),
                     ("Val-train MAE gap, logged","gap_val_minus_train_mae_logged"),
                     ("Training seconds, logged","training_seconds_logged")]:
    b_rows.append((display,f'{m["delta1"][field]:.6f}',f'{m["delta2"][field]:.6f}'))
b_rows.extend([("ML>=4.5 underprediction","75/75 (100%)","75/75 (100%)"),("Best/total epochs","12/20","11/19")])
b_text=f"""# Phase 2B Huber experiment — corrected report, revision 1

**Phase 2E revision, 2026-11-10.** Supersedes the narrative and labels in [the original report](huber_experiment_report.md), which remains unchanged. Statistics reuse [independent Phase 2D verification](../gemini_verification/verification_report.md); additional checkpoint-metadata evidence is recorded in Phase 2E. No new training, inference or model selection.

## Verified results and corrections

Targets use original ML units and the identity transform. Baseline delta=1, new delta=2; validation n=1065.

{table(["Metric","delta1, Variant C","delta2, Variant C"],b_rows)}

MAE improves by {changes["mae_delta2_minus_delta1"]:.6f} ML ({changes["mae_improvement_percent"]:.4f}%), but prediction SD falls {changes["prediction_sd_reduction_percent"]:.4f}% and high-magnitude MAE worsens {changes["high_mae_change"]:.6f} ML. All75 high targets remain underpredicted; RMSE/R2 do not improve. This is a modest validation tradeoff, not established generalization improvement.

- Delta2 Huber loss is **{m["delta2"]["huber"]:.10f}**, replacing0.2312. Different delta thresholds define different loss objectives; their raw values are not equivalent quality scores.
- Prediction mean moves **farther from** actual mean {m["delta2"]["actual_mean"]:.6f}: absolute offsets {m["delta1"]["mean_distance_from_actual"]:.6f} -> {m["delta2"]["mean_distance_from_actual"]:.6f}.
- Delta2 prediction SD is **{100*m["delta2"]["predicted_actual_sd_ratio"]:.3f}% of actual SD**, while variance is **{100*m["delta2"]["predicted_actual_variance_ratio"]:.3f}% of actual variance**. The original19.8% variance description was incorrect.
- Recorded delta2 runtime is **{m["delta2"]["training_seconds_logged"]:.6f}s**, replacing10.3s; baseline is {m["delta1"]["training_seconds_logged"]:.6f}s. Saved timings are not an independent latency benchmark or proof of identical efficiency.
- The smaller train/validation gap accompanies worse training MAE and more compression; it does not establish reduced overfitting across different distributions.
- Compression increased in this fit, but its causal mechanism was not established. Gradients use weighted training observations, not the validation majority. Huber optimization does not necessarily produce constant predictions or the arithmetic mean.

Statistical reliability remains uncertain: one seed,75 high validation targets, overlapping histories and checkpoint selection on the same validation set. The retained RMSE figure uses the correct fallback for this baseline history; its conditional plotting code is fragile, not evidence of an incorrect saved curve.

## Strengthened control evidence and remaining limits

**PASS — serialized settings:** Both best checkpoints retain matching hyperparameters, model configuration, dataset hash/version, Variant C, target definition, seed42,538881 parameters and initialization hash. Saved losses are {a["loss"]} versus {b["loss"]}. Full-precision settings: hidden128, layers2, dropout {a["hyperparameters"]["dropout"]}, learning rate {a["hyperparameters"]["learning_rate"]}, weight decay {a["hyperparameters"]["weight_decay"]}, batch128. Last-checkpoint AdamW parameter groups match, including betas(0.9,0.999), epsilon1e-8 and optional flags.

**PASS — recorded selection:** Best-checkpoint epochs12/11 and validation MAE match history minima and saved predictions. Delta2 last checkpoint records epoch19, best epoch11, eight stale epochs. Phase 2D independently verified identical initialization and all19 common sampled-parent/order hashes. Supplied notebook code selects by validation MAE and indexes training/validation tensors only.

**UNKNOWN — full historical execution:** Checkpoint metadata adds evidence beyond Phase 2D's source inspection, but no complete delta2 runtime/environment snapshot or independent access log establishes every execution fact. Its checkpoint omits the consumer-source hash. Exact full runtime equivalence and absolute historical test non-access remain UNKNOWN; intended validation-only selection is supported.

Metadata was read with restricted fake tensors, without materializing checkpoint tensor storage or constructing models. Original reports, predictions, metrics, histories, figures and checkpoints are unchanged. The original notebook has no safe resume logic and would overwrite its own delta2 artifacts if rerun.

Evidence: [correction summary](../gemini_verification/correction_report.md), [metadata/search record](../gemini_verification/correction_provenance_v1.json).
"""
feature_table=table(["Index","Feature","Units","Train mean +/- SD","Validation mean +/- SD"],[
    (f["index"],f["name"],f["units"],f'{f["tr_mean"]:.6f} +/- {f["tr_std"]:.6f}',
     f'{f["val_mean"]:.6f} +/- {f["val_std"]:.6f}') for f in features])
correction_table=table(["Index","Feature","Field","Original","Verified replacement"],[
    (c["index"],c["name"],c["field"],c["reported"],f'{c["recomputed"]:.8f}') for c in corrections])
corr_table=table(["Index","Pearson H1","Pearson H2","Pearson validation","Spearman train","Spearman validation"],[
    (f["index"],f'{f["r_h1"]:.6f}',f'{f["r_h2"]:.6f}',f'{f["r_val"]:.6f}',f'{f["rho_train"]:.6f}',f'{f["rho_val"]:.6f}') for f in features])
s=verified["phase2c_temporal"]["split_summaries"]
periods=verified["phase2c_features"]["reconstructed_periods"]
dist=verified["phase2c_temporal"]["distances"]
split_table=table(["Split","n","Mean target ML","ML>=4.5 count (%)","N/elapsed year","Mean30-day rate"],[
    (name,v["n"],f'{v["target_mean"]:.6f}',f'{v["high_count"]} ({v["high_percent"]:.3f}%)',
     f'{v["targets_per_elapsed_year_N_over_span"]:.3f}',f'{v["last_input_rate_30d_mean"]:.6f}')
    for name,v in s.items()])
c_text=f"""# Phase 2C feature and temporal audit — corrected report, revision 1

**Phase 2E revision, 2026-11-10.** Supersedes the summaries and interpretations in [the original report](phase2c_feature_informativeness_report.md). Original JSON, figures and report remain unchanged. All numbers reuse independently verified Phase 2D statistics; no data reprocessing, training or inference.

## Corrected feature table

Final historical step19, inverse-scaled to documented units; population SD. Train4973, validation1065, test1067. This is a retrospective next-retained-ML-event catalog diagnostic, not operational EEW or demonstrated earthquake forecasting.

{feature_table}

All40 mean/SD entries are shown consistently above. The19 erroneous original entries are corrected explicitly below; fields tr/val denote training/validation.

{correction_table}

## Correlations and definitions

{corr_table}

Phase 2D reproduced all340 original JSON numeric fields. Matching reconstructed periods: H1 rows[0,2486), n2486, target origins {periods["train_h1"]["first_target"]} through {periods["train_h1"]["last_target"]}; H2 rows[2486,4973), n2487, {periods["train_h2"]["first_target"]} through {periods["train_h2"]["last_target"]}. These are **reconstructed boundaries, not recovered original execution provenance**.

Rolling-mean r decreases0.826588 ->0.290506 ->0.200656. The last value exceeds0.20, so the categorical original threshold claim is withdrawn. Changes in individual correlations describe instability but do not prove absence of nonlinear, joint or sequence predictive signal. Squared univariate r is not joint or held-out model explained variance.

Training redundancy pairs are features0/6 (r0.818387) and8/9 (r0.657882). The original1/8 pairing is incorrect (actual r0.410652). Training gap Spearman is +0.396137, while event-rate Spearman is -0.599379; both are not negative.

Original d_train/d_val fields are **group mean differences divided by overall-split SD**, not pooled-within-group Cohen d. Keep their stored values under this corrected interpretation. Example: feature0 training value {features[0]["d_train"]:.6f}, versus independently verified pooled Cohen d {features[0]["cohens_d_pooled_train"]:.6f}. The verified feature CSV retains both definitions.

## Distribution shifts and frequency limits

{split_table}

Annual rates use exact first-to-last target elapsed seconds and365.2425days/year: spans100.353619/1.128955/0.663893years. Validation/test rates943.350/1607.187 replace942/1592 calculated from rounded durations. Alternative(N-1)/span interarrival rates are saved in the verified JSON. Mean30-day test/train ratio is8.1119, replacing8.2 from rounded inputs.

Counts and rates concern **retained ML events**, not all source events or complete regional seismicity. Catalog downtime, missing periods, reporting policies and actual exposure are unverified. Exact endpoint durations account for partial years but do not establish coverage. The30-day feature counts retained events in(time-30days,time] divided by30. Final-input gap medians are1.556725/0.197836/0.131713days; these differ in definition from separately verified target-event gaps.

Train-to-validation Wasserstein={dist["val"]["wasserstein"]:.6f}, KS D={dist["val"]["ks_statistic"]:.6f}, nominal p={dist["val"]["ks_nominal_pvalue"]:.4g}; train-to-test Wasserstein={dist["test"]["wasserstein"]:.6f}. These descriptive distances reproduce. Conventional KS/Pearson/Spearman p values are not dependence-robust here: overlapping histories, temporal structure and tied ML observations limit significance claims. Saved zero p values reflect numerical underflow.

## Confirmed findings versus hypotheses

**Confirmed:** temporal target/feature distribution changes; correlation weakening/sign changes; compressed model predictions and high-magnitude underprediction. Input-before-target ordering was verified; origin times do not establish historical reporting availability.

**Hypotheses only:** changing detection/reporting, spatial sampling, magnitude-type selection and physical clustering may contribute. No causal completeness analysis, validated station-network deployment history or learned-rule analysis establishes the proposed causes.

**Historical Mc UNKNOWN; current Mc UNKNOWN.** No formal validated estimator, fit diagnostics or uncertainty intervals were supplied. Claims Mc~4.0/~1.5 or that only ML>=4.0 events were historically detected are withdrawn as established findings. Observed minimum magnitudes are not Mc estimates. Modern catalog stationarity is unverified.

The Gutenberg-Richter frequency-magnitude relationship does **not** prove independent successive magnitudes or conditional independence. Small individual correlations do not prove no predictive signal, nor that Huber regression must predict the mean. The [Phase 2D report](../gemini_verification/verification_report.md) retains the methodological reference and scientific qualifications; no new completeness or independence study was introduced.

## Original Phase 2C execution provenance: missing

Targeted filename/title/content searches within this project, including hidden notebook locations and excluding dependency runtime, found no original Phase 2C notebook. Immediate workspace and named notebook-checkpoint locations yielded no matching recovery copy. The alternate content hit is the existing Phase 2A diagnosis, not Phase 2C. No machine-wide search is claimed.

Original statistics remain preserved, with independent reproducibility supported by the executed Phase 2D notebook. **Neither that verification nor this correction notebook is a replacement execution history.** Original figures are preserved; their generation provenance is missing. Registered checkpoints are unchanged; unrecorded Phase 2C training cannot be ruled out.

Evidence: [correction summary](../gemini_verification/correction_report.md), [provenance record](../gemini_verification/correction_provenance_v1.json), [verified feature statistics](../gemini_verification/feature_verification.csv).
"""
b_revision=HUB/"huber_experiment_report_revision_v1.md"
c_revision=DIAG/"phase2c_feature_informativeness_report_revision_v1.md"
b_revision.write_text(b_text,encoding="utf-8")
c_revision.write_text(c_text,encoding="utf-8")
for c in corrections:
    assert f'{c["recomputed"]:.8f}' in c_text
assert len(corrections)==19 and all(protected[rel]==sha(ROOT/rel) for rel in protected)
provenance["protected_hashes_after"]={rel:sha(ROOT/rel) for rel in protected}
provenance["source_preservation_pass"]=True
provenance["corrected_feature_summary_count"]=19
provenance["all_19_replacements_present"]=True
provenance["revisions"]={str(p.relative_to(ROOT)):sha(p) for p in [b_revision,c_revision]}
(OUT/"correction_provenance_v1.json").write_text(json.dumps(provenance,indent=2,ensure_ascii=True),encoding="utf-8")
summary="""# Phase 2E correction report

Date: 2026-11-10. Reporting/provenance work only: no training, inference, optimization, dataset loading/regeneration or CNN work.

**Documentation: READY WITH LIMITATIONS for planning the next experiment.** Corrections are complete and limitations explicit. This does not certify complete historical reproducibility or authorize another experiment.

## Versioned corrections

- [Phase 2B revision 1](../huber_experiment/huber_experiment_report_revision_v1.md): corrected delta2 Huber loss to 0.1902281602; SD ratio to 19.829% versus variance ratio 3.932%; prediction mean moves farther from the validation mean; runtime 11.078315s versus baseline 10.753820s. Overfitting, significance and compression-mechanism claims are qualified.
- [Phase 2C revision 1](../error_diagnosis/phase2c_feature_informativeness_report_revision_v1.md): complete corrected feature table and all 19 original/replacement entries. Corrected redundancy pairing, effect-size label, correlation-threshold overstatement and annual-rate approximations.
- Confirmed distribution shifts are separated from completeness, station-network and learned-rule hypotheses. Historical/current Mc are UNKNOWN. Gutenberg-Richter statistics do not prove independent magnitudes; significance is qualified for temporal dependence.
- Original reports, numerical artifacts, figures, checkpoints and Phase 2D verification are preserved. Cite the versioned revisions for current interpretation.

## Additional provenance evidence

**PASS — saved controls:** restricted fake-tensor metadata inspection confirms matching full-precision hyperparameters, model configuration, v2 dataset hash, Variant C, identity target definition, seed, parameter count and initialization hash. Last-checkpoint AdamW parameter groups match. Best-checkpoint epochs 12/11 and validation MAE agree with history minima; delta2 last state records epoch 19 and eight stale epochs. No model was constructed or run.

**UNKNOWN — full execution guarantees:** complete delta2 runtime/environment and independent test-access logs remain unavailable; its checkpoint omits the augmentation consumer-source hash. Supplied code supports validation-only selection, but exact full historical execution equivalence and absolute test non-access cannot be certified.

**NOT FOUND within targeted scope — original Phase 2C notebook:** no matching notebook or checkpoint copy was recovered. The content-search candidate is Phase 2A. Preserve verified statistics and independent Phase 2D reproduction separately; original execution history remains missing.

## Before the next experiment

Use revised reports and explicitly retain unresolved provenance limits. A future run needs a complete configuration/environment snapshot, source hashes, split/selection evidence and resumable execution log. Any causal-completeness assertion requires a separately justified analysis; none was performed here.

Evidence: [provenance/preservation record](correction_provenance_v1.json), [independent verification](verification_report.md), [executed Phase 2E notebook](../../phase2e_report_corrections.ipynb).
"""
(OUT/"correction_report.md").write_text(summary,encoding="utf-8")
marker="## Phase 2E report corrections and provenance — 2026-11-10"
note=f"""{marker}

- READY WITH LIMITATIONS for next-experiment planning; no training, inference or dataset changes.
- Versioned Phase 2B/C revisions correct all 19 feature summaries plus Huber loss, variance, mean shift, runtime, labels and scientific interpretations. Original artifacts preserved.
- Saved checkpoint configurations/AdamW groups and validation-selected epochs match. Complete historical execution equivalence and absolute test non-access remain UNKNOWN.
- Original Phase 2C notebook not recovered; verified statistics preserved separately. Historical/current Mc UNKNOWN; completeness causation remains a hypothesis.
- Current documentation: phase2_bilstm_results/gemini_verification/correction_report.md. Earlier Gemini Phase 2B/C narrative entries below are superseded by these revisions and retained only as historical notes.

"""
for p in [ROOT/"project_status.md",ROOT/".agent-memory/checkpoint.md"]:
    old=p.read_text(encoding="utf-8")
    if marker not in old:
        p.write_text(note+old,encoding="utf-8")
assert all(protected[rel]==sha(ROOT/rel) for rel in protected)
for p in [b_revision,c_revision,OUT/"correction_report.md",OUT/"correction_provenance_v1.json"]:
    assert p.is_file() and p.stat().st_size>0
print("Corrected 19/19 summaries; report revisions, provenance record and status notes saved.")
print("Matching checkpoint configurations and AdamW groups: PASS; preserved sources:",len(protected))
print("Candidate:",candidate_reviews)


Corrected 19/19 summaries; report revisions, provenance record and status notes saved.
Matching checkpoint configurations and AdamW groups: PASS; preserved sources: 57
Candidate: [{'path': 'phase2a_bilstm_error_diagnosis.ipynb', 'title': '# Phase 2A — saved-output diagnosis only\n\nSelected C is fixed. No checkpoint loading, inference, training, Optuna, preprocessing, loss/sampler changes or model selection. Saved predictions are sufficient. Test distributi', 'matching_lines': ['## Loss and feature informativeness'], 'classification': 'Phase 2A artifact, not recovered Phase 2C execution'}]
